In [16]:
from dotenv import load_dotenv
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [19]:
from langchain.tools import tool,ToolRuntime
@tool
def read_email(runtime: ToolRuntime):
    """read on email from the given address"""
    return runtime.state['email']

@tool
def send_email(body:str):
    """send emaail to the given address with given subject and  body """
    return f"email sent"

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
class EmailState:
    email:str


agent=create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email":False,
                "send_email":True,
            },
            description_prefix="tool execution require approval"
        )
    ],
)

In [33]:
from langchain.messages import HumanMessage
config={"configurable":{"thread_id":"4"}}
response=agent.invoke({
    "messages":[HumanMessage(content="please read my email and send response immedatily.send reply now in the same thread")],
    "email":"Hi sean Iam going ti be late for our meeting tommorrow.can you reschedule?Best John,"
    },
    config=config
)

In [34]:
from pprint import pprint
for message in response['messages']:
    print("=======================================")
    pprint(message)

HumanMessage(content='please read my email and send response immedatily.send reply now in the same thread', additional_kwargs={}, response_metadata={}, id='4ea7742e-0cf7-4395-882a-3d3d5d2fcef9')
AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gygtew2b5', 'function': {'arguments': '{}', 'name': 'read_email'}, 'type': 'function'}, {'id': 'q4nhb5w6e', 'function': {'arguments': '{"body":"I have read your email and I will respond in the next few hours.","subject":"Re: your email","to":"your_email@example.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 290, 'total_tokens': 344, 'completion_time': 0.102169295, 'completion_tokens_details': None, 'prompt_time': 0.017075875, 'prompt_tokens_details': None, 'queue_time': 0.158849083, 'total_time': 0.11924517}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logp

In [27]:
from langgraph.types import Command
response=agent.invoke(
    Command(
        resume={"decisions":[{"type":"approve"}]}
    ),
    config=config
)
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          "I've "
                                                                          'rescheduled '
                                                                          'the '
                                                                          'meeting '
                                                                          'for '
                                                                          '2 '
                                                                          'PM. '
                                                                          'Looking '
                                                                          'forward '
                                                                          'to '
         

In [28]:
response=agent.invoke(
    Command(
        resume={"decisions":[
            {
                "type":"reject",
                "message":"No please signoff your merciful leader Sean"
                              }
        ]
        }
    ),
    config=config
)
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Respectfully, '
                                                                          'you '
                                                                          'will '
                                                                          'not '
                                                                          'be '
                                                                          'late. '
                                                                          'I '
                                                                          'am '
                                                                          'your '
                                                                          'merciful '
                                                                          'leader. '
                                                                          '-Sean',
       

In [35]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send-email",  # tool name — must match the tool exactly
                        "args": {
                            "body": "This is the last straw, you are fired"
                        },
                    },
                }
            ]
        }
    ),
    config=config,
)
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': "Let's "
                                                                          'reschedule '
                                                                          'for '
                                                                          '2 '
                                                                          'PM '
                                                                          'instead',
                                                                  'subject': 'Rescheduled '
                                                                             'Meeting',
                                                                  'to': 'john@example.com'},
                                                         'description': 'tool '
                                                                        'execution '
                                                                  